# B5/B6/B7 — Review variant_claims candidates, guarantee floor conflicts, measure extraction quality

Per `docs/TODO-stage4.md` Track B:
- **B5**: review every row in `variant_claims_candidates.json` against its source
  `passage_ref`; promote approved rows by setting `trust_tier=1`.
- **B6**: guarantee the seed contains all three floor conflicts (Aphrodite parentage,
  Io parentage, Achilles death) regardless of what extraction found — hand-add only
  what extraction missed.
- **B7** (diagnostic, non-blocking): measure how many of the two *measurable* floor
  conflicts (Aphrodite, Achilles — Io is structurally excluded, see below) the pipeline
  detected unaided.

Run B2 (`python -m extraction.run_extraction`) first if you haven't — this notebook
reads its output from `extraction/output/variant_claims_candidates.json` and writes
back to the same file (never a second copy — `trust_tier` is the only thing that
changes on review).

In [12]:
import json
import sys
from pathlib import Path

INGESTION_ROOT = Path.cwd().parent
sys.path.insert(0, str(INGESTION_ROOT))

from dotenv import load_dotenv
load_dotenv()

CANDIDATES_PATH = INGESTION_ROOT / "extraction" / "output" / "variant_claims_candidates.json"

In [13]:
from loader.source_registry import SOURCE_REGISTRY
from loader.text_cleaner import clean
from extraction.segmentation import segment

candidates = json.loads(CANDIDATES_PATH.read_text(encoding="utf-8"))
print(f"{len(candidates)} candidate rows loaded")

7429 candidate rows loaded


## B7 — Extraction-quality metric (diagnostic, non-blocking)

Measured against the raw candidates, **before** any B6 hand-add below. Only Aphrodite
and Achilles are measurable this way (`N/2`, not `N/3`) — Io's two variants are both
attributed to Apollodorus (a single `source_id`), so the `count(DISTINCT source_id) >=
2` detector can never emit it; that's expected, not a pipeline miss.

In [14]:
def floor_conflict_detected(subject: str, claim_type: str) -> bool:
    rows = [
        c for c in candidates
        if c["subject_name"].strip().lower() == subject.lower() and c["claim_type"] == claim_type
    ]
    return len({c["source_id"] for c in rows}) >= 2


aphrodite_detected = floor_conflict_detected("Aphrodite", "parentage")
achilles_detected = floor_conflict_detected("Achilles", "death")
detected_count = sum([aphrodite_detected, achilles_detected])

print(f"floor conflicts detected: {detected_count}/2")
print(f"  Aphrodite parentage: {'detected' if aphrodite_detected else 'MISSED'}")
print(f"  Achilles death:      {'detected' if achilles_detected else 'MISSED'}")
print("  Io parentage:        structurally excluded (single-source, not measurable)")

floor conflicts detected: 2/2
  Aphrodite parentage: detected
  Achilles death:      detected
  Io parentage:        structurally excluded (single-source, not measurable)


## B5 — Review every candidate row, grouped by disagreement

Dumping all 7,429 rows in one cell floods Jupyter's IOPub rate limit (and isn't readable anyway). The real review unit is the **841 distinct `(subject, claim_type)` groups** -- each one disagreement, with 1+ candidate rows supporting each side. Review one group at a time with `review_group(subject, claim_type)` below.

For each row shown: check the `claim_value` against the segment text -- the exact text the LLM saw, re-derived the same way `run_extraction.py` builds it (**not** a `narrative_chunks` lookup, which uses different chunk boundaries) -- and decide whether it's a real, correctly-attributed claim.

In [15]:
def build_segment_map(source):
    raw = (INGESTION_ROOT / source.file_path).read_text(encoding="utf-8")
    cleaned = clean(raw)
    segs = segment(cleaned, source.author, source.work, source.passage_ref_extractor)
    return {s.passage_ref: s.text for s in segs}


segment_maps = {s.source_id: build_segment_map(s) for s in SOURCE_REGISTRY}

In [16]:
from collections import defaultdict

groups: dict[tuple[str, str], list[int]] = defaultdict(list)
for i, c in enumerate(candidates):
    groups[(c["subject_name"].strip().lower(), c["claim_type"])].append(i)

print(f"{len(groups)} distinct (subject, claim_type) groups across {len(candidates)} candidate rows")
print("(one line per group -- pick a subject/claim_type below and call review_group(...) to see its rows)\n")
for (subject, claim_type), idxs in sorted(groups.items()):
    distinct_sources = len({candidates[i]["source_id"] for i in idxs})
    print(f"  {subject!r:30} {claim_type!r:20} {len(idxs)} rows, {distinct_sources} distinct sources")

841 distinct (subject, claim_type) groups across 7429 candidate rows
(one line per group -- pick a subject/claim_type below and call review_group(...) to see its rows)

  ''                             'parentage'          4 rows, 2 distinct sources
  '<none>'                       'parentage'          3 rows, 2 distinct sources
  '<unknown>'                    'parentage'          101 rows, 6 distinct sources
  'abas'                         'parentage'          4 rows, 2 distinct sources
  'acamas'                       'parentage'          7 rows, 2 distinct sources
  'achelous'                     'parentage'          4 rows, 2 distinct sources
  'achilles'                     'burial'             2 rows, 2 distinct sources
  'achilles'                     'death'              34 rows, 4 distinct sources
  'achilles'                     'epithet'            4 rows, 2 distinct sources
  'achilles'                     'marriage'           2 rows, 2 distinct sources
  'achilles'      

In [27]:
def review_group(subject: str, claim_type: str) -> None:
    """Prints every candidate row for one (subject, claim_type) group, with its
    source segment text. Call this repeatedly for whichever groups you're reviewing
    right now -- output stays small per call, so no IOPub flooding. Ends with a
    ready-to-paste `approved_indices += [...]` line covering every index shown --
    paste it below, delete whichever indices you're rejecting, then run."""
    key = (subject.strip().lower(), claim_type)
    idxs = groups.get(key)
    if not idxs:
        print(f"no group for {subject!r}/{claim_type!r} -- check the listing above for the exact claim_type spelling")
        return
    for i in idxs:
        c = candidates[i]
        print("=" * 80)
        print(f"[{i}] subject={c['subject_name']!r}  claim_type={c['claim_type']!r}")
        print(f"    claim_value: {c['claim_value']}")
        print(f"    source_id={c['source_id']}  passage_ref={c['passage_ref']}  trust_tier={c['trust_tier']}")
        segment_text = segment_maps.get(c["source_id"], {}).get(c["passage_ref"])
        if segment_text is None:
            print("    [!] no matching segment for this passage_ref -- verify manually against the source text")
        else:
            # preview = segment_text[:400] + ("..." if len(segment_text) > 400 else "")
            preview = segment_text
            print(f"    segment text: {preview}")
    print("-" * 80)
    print(f"copy/paste (delete any indices you're rejecting): approved_indices += {idxs}")


# Example -- start with the floor conflicts:
review_group("Io", "parentage")

[2590] subject='Io'  claim_type='parentage'
    claim_value: child of Iasus
    source_id=apollodorus-bibliotheca  passage_ref=2.1.2-2.1.3  trust_tier=1
    segment text: About him I shall speak again. But Argus received the kingdom and called the Peloponnese after himself Argos; and having married Evadne, daughter of Strymon and Neaera, he begat Ecbasus, Piras, Epidaurus, and Criasus, who also succeeded to the kingdom.
Ecbasus had a son Agenor, and Agenor had a son Argus, the one who is called the All-seeing. He had eyes in the whole of his body, and being exceedingly strong he killed the bull that ravaged Arcadia and clad himself in its hide; and when a satyr wronged the Arcadians and robbed them of their cattle, Argus withstood and killed him. It is said, too, that Echidna, daughter of Tartarus and Earth, who used to carry off passers-by, was caught asleep and slain by Argus. He also avenged the murder of Apis by putting the guilty to death.
Argus and Ismene, daughter of Asopus, had

## Decide which rows to promote

Edit the list below with the indices (from the `[i]` printed by `review_group`) you approve this session -- run it, then repeat across as many sessions as you like. **Additive only**: this only ever sets rows to `trust_tier=1`, never demotes an already-promoted row back down, so partial review across multiple sittings is safe -- you don't need to re-list indices you approved in an earlier session.

In [18]:
# <-- EDIT THIS after reviewing the rows above
approved_indices: list[int] = []

In [19]:
approved_indices += [743, 744, 745, 746, 747, 748,  750, 752, 753, 754, 755, 756, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766, 767, 768, 769, 770, 771, 772, 773, 774, 775, 776, 777, 778, 779, 780]

In [22]:
approved_indices += [2590, 2591, 2592, 2593, 2594, 2595, 2596, 2597, 2598, 2599]

In [24]:
approved_indices += [5759, 5760, 5761, 5762, 5764,  5766, 5772, 5773, 5774, 5775, 5776, 5777, 5778, 5779, 5780, 5781, 5782, 5783, 5784, 5785, 5786, 5787, 5788, 5789, 5790, 5791, 5792]

In [25]:
for i in approved_indices:
    candidates[i]["trust_tier"] = 1

CANDIDATES_PATH.write_text(json.dumps(candidates, indent=2, ensure_ascii=False), encoding="utf-8")
total_promoted = sum(1 for c in candidates if c["trust_tier"] == 1)
print(f"{len(approved_indices)} row(s) confirmed this run; {total_promoted} total rows now at trust_tier=1 (out of {len(candidates)})")

73 row(s) confirmed this run; 73 total rows now at trust_tier=1 (out of 7429)


## B6 — Guarantee the three floor conflicts

Check which floor conflicts are already covered by promoted (`trust_tier=1`) rows.
Extraction-preferred: only hand-add the ones missing below.

In [26]:
def floor_conflict_covered(subject: str, claim_type: str, min_distinct_values: int = 2) -> bool:
    promoted = [
        c for c in candidates
        if c["trust_tier"] == 1
        and c["subject_name"].strip().lower() == subject.lower()
        and c["claim_type"] == claim_type
    ]
    return len({c["claim_value"] for c in promoted}) >= min_distinct_values


coverage = {
    "Aphrodite parentage": floor_conflict_covered("Aphrodite", "parentage"),
    "Io parentage": floor_conflict_covered("Io", "parentage"),
    "Achilles death": floor_conflict_covered("Achilles", "death"),
}
for name, covered in coverage.items():
    print(f"{name}: {'covered' if covered else 'MISSING -- hand-add below'}")

Aphrodite parentage: covered
Io parentage: covered
Achilles death: covered


## Hand-add any floor conflict extraction missed

**Keep the `claim_type` canonical** (per DEV-018/DEV-020: Achilles' is `death`, not
`slaying`/`death_manner`) so a hand-added row groups with any extracted counterpart and
the exact-match `ConflictLookup` returns both. Run the normalize check first to
confirm the value you're about to use.

In [ ]:
import psycopg2

import config
from extraction.claim_type_normalizer import load_alias_map, normalize

conn = psycopg2.connect(
    host=config.POSTGRES_HOST, port=config.POSTGRES_PORT, dbname=config.POSTGRES_DB,
    user=config.POSTGRES_USER, password=config.POSTGRES_PASSWORD,
)
alias_map = load_alias_map(conn)
conn.close()

print(normalize(alias_map, "slaying"))    # -> "death"
print(normalize(alias_map, "parent_of"))  # -> "parentage"

In [ ]:
# Edit and run only for floor conflicts flagged MISSING above. Example shown for Io
# (Inachus vs Piren, per Apollodorus -- confirm the real passage_ref before running).
hand_added = [
    {
        "subject_name": "Io",
        "claim_type": "parentage",
        "claim_value": "daughter of Inachus",
        "source_id": "apollodorus-bibliotheca",
        "passage_ref": "2.1.3",
        "trust_tier": 1,
    },
    {
        "subject_name": "Io",
        "claim_type": "parentage",
        "claim_value": "daughter of Piren",
        "source_id": "apollodorus-bibliotheca",
        "passage_ref": "2.1.3",
        "trust_tier": 1,
    },
]

candidates.extend(hand_added)
CANDIDATES_PATH.write_text(json.dumps(candidates, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"{len(hand_added)} rows hand-added; {len(candidates)} total rows now in the file")

## Final check

Re-run the B6 coverage cell above after any hand-adds — all three floor conflicts
must show `covered` before handing this off for Track C (`V12`).